# What is Jev? TypeSafe's System One model, paired with Claude

The notebook from the video, two ways to run the same agent.

The agent answers support questions from eight read-only lookups. Each lookup has a simulated 1.5 s round trip.

1. **Claude alone** (`agent_claude.py`): Claude Sonnet 5 picks each lookup, waits for it, and repeats. Three of every four Claude calls only decide what to look up next.
2. **Jev fetches, Claude writes** (`agent_jev_first.py`): Jev scores every possible lookup in one call, the harness runs the likely ones, and Claude gets the facts in a single call with no tools and no schemas. If a fact is missing, Claude says so and the run falls back to the tool-using agent.


In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt

from agent_claude import MODEL, run_agent
from agent_jev_first import run_agent_jev_first

REPEATS = 3
tasks = json.load(open("tasks.json"))
task = tasks[0]["task"]
print(MODEL)
task

claude-sonnet-5


'Customer C-17 says their gateway order never arrived. Find out what happened to it and whether they can get a refund.'

## One task, Claude alone

In [2]:
base = await run_agent(task)
print(f"{base.seconds:.1f}s   claude calls={base.claude_calls}   lookups={base.tool_calls}   input tokens={base.input_tokens}   ${base.claude_usd:.4f}\n")
print(base.answer)

12.1s   claude calls=4   lookups=6   input tokens=5997   $0.0179

Order O-1042 (the gateway, $1,840, paid) is currently stuck — DHL shows it held at customs in Leipzig since Sept 9, with no ETA, and it's been over 10 days in transit. Under EU policy, since it's exceeded the 10-day transit threshold, Priya is entitled to choose either a full refund or a reshipment. I'd recommend contacting her to confirm which option she prefers.


## The same task, Jev fetches and Claude writes

In [ ]:
first, s, fell_back = await run_agent_jev_first(task)
print(f"{first.seconds:.1f}s   claude calls={first.claude_calls}   lookups={first.tool_calls}   input tokens={first.input_tokens}   ${first.claude_usd:.4f} + Jev ${s.jev_usd:.5f}   fell back={fell_back}\n")
print(first.answer)

6.1s   claude calls=1   lookups=7   input tokens=513   $0.0024 + Jev $0.00012   fell back=False

Order O-1042 (the gateway, SKU-301) is stuck in DHL customs since Sept 9 in Leipzig with no ETA — it hasn't been lost, just delayed over 10 days in transit. Per EU policy, since it's been stuck longer than 10 days, Priya is entitled to choose either a full refund or a reshipment. Please confirm with her which option she prefers so we can process it.


## All tasks, both ways, repeated

In [7]:
rows = []
for rep in range(REPEATS):
    for item in tasks:
        base = await run_agent(item["task"])
        first, s, fell_back = await run_agent_jev_first(item["task"])
        for arm, run, jev_usd, fb in [("Claude alone", base, 0.0, False), ("Jev fetches first", first, s.jev_usd, fell_back)]:
            rows.append({
                "arm": arm, "task": item["id"], "rep": rep, "seconds": run.seconds,
                "claude_calls": run.claude_calls, "input_tokens": run.input_tokens, "lookups": run.tool_calls,
                "claude_usd": run.claude_usd, "jev_usd": jev_usd, "usd": run.claude_usd + jev_usd, "fell_back": fb,
            })
df = pd.DataFrame(rows)
df.to_csv("runs.csv", index=False)
len(df)

48

### Median seconds per task

In [8]:
ARMS = ["Claude alone", "Jev fetches first"]
seconds = df.pivot_table(index="task", columns="arm", values="seconds", aggfunc="median")[ARMS]
seconds["speedup"] = seconds["Claude alone"] / seconds["Jev fetches first"]
seconds.round(2)

arm,Claude alone,Jev fetches first,speedup
task,,,
account-review,8.29,5.62,1.47
backorder,7.17,5.17,1.39
dead-switch,8.27,6.17,1.34
double-charge,8.28,5.34,1.55
in-transit,12.33,3.43,3.59
late-gateway,11.73,5.89,1.99
policy-only,4.33,3.16,1.37
unpaid,8.86,7.37,1.20


In [ ]:
ax = seconds[ARMS].plot.bar(figsize=(10, 4), color=["#9a9a9a", "#c4622d"], rot=30)
ax.set_ylabel("median seconds per task")
plt.tight_layout()

### Claude calls, tokens and cost per task (medians)

In [9]:
cost = df.pivot_table(index="arm", values=["seconds", "claude_calls", "input_tokens", "lookups", "claude_usd", "jev_usd", "usd"], aggfunc="median").loc[ARMS]
cost[["seconds", "claude_calls", "input_tokens", "lookups", "claude_usd", "jev_usd", "usd"]].round({"seconds": 2, "claude_usd": 5, "jev_usd": 6, "usd": 5})

,seconds,claude_calls,input_tokens,lookups,claude_usd,jev_usd,usd
arm,,,,,,,
Claude alone,8.39,3.0,4214.5,4.0,0.01277,0.00000,0.01277
Jev fetches first,5.37,1.0,350.0,4.5,0.00206,0.00011,0.00216


In [ ]:
ax = cost[["claude_usd", "jev_usd"]].plot.bar(stacked=True, figsize=(6, 4), color=["#9a9a9a", "#c4622d"], rot=0)
ax.set_ylabel("median USD per task")
ax.legend(["Claude", "Jev"])
plt.tight_layout()

In [ ]:
total = df.groupby("arm")[["seconds", "usd"]].sum().loc[ARMS] / REPEATS
fallbacks = int(df[df.arm == "Jev fetches first"].fell_back.sum())
print(f"Time   {total.seconds['Claude alone']:.1f}s -> {total.seconds['Jev fetches first']:.1f}s   ({total.seconds['Claude alone'] / total.seconds['Jev fetches first']:.2f}x faster)")
print(f"Cost   ${total.usd['Claude alone']:.4f} -> ${total.usd['Jev fetches first']:.4f}   ({total.usd['Claude alone'] / total.usd['Jev fetches first']:.1f}x cheaper)")
print(f"Fell back to the tool-using agent in {fallbacks} of {len(tasks) * REPEATS} runs")

## Read this before trusting the numbers

- Medians over 3 runs. Claude takes different paths from run to run, and one slow response can flip a task.
- Tool latency is simulated at 1.5 s. Jev-first removes Claude round trips, so its gain grows as tools get faster.
- Only read-only lookups are fetched this way. Never let a predictor run a write.
- Jev reads the question literally. Asking "is X the very next lookup" halved its accuracy, because parallel lookups split the probability. Asking "is X one of the lookups to run now" fixed it.
- Claude cannot ask for more. The MISSING fallback catches under-fetching. It cannot catch a wrong answer written from incomplete facts.